# 24 — Family 4 (F4): Query-Based Transformer Detector (`RT-DETR-L`)

**Experiments under Fixed Protocol P0 (`k5_drop_L1920`, `1.5 GPU-h`, `imgsz=1280`):**
- **`rtdetr-e0` (Baseline):** COCO-pretrained `rtdetr-l.pt`, `nq=300` object queries, AdamW, `1.5 GPU-h` budget.
- **`rtdetr-e1` (Exp A — Query Density `nq=600`):** Expanding decoder queries from `300 -> 600` (`configs/ultralytics_models/rtdetr-l-nq600.yaml`) for dense VisDrone scenes (up to 180 objects/frame).
- **`rtdetr-e2` (Exp B — Hungarian Matching Cost Gains & Contrastive Denoising):** Re-weighting Hungarian bipartite matching (`bbox=7.5, giou=3.0, class=1.5`) and doubling denoising groups (`100 -> 200`).

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from vdet.paths import P
from vdet.utils import seed_everything
from vdet.tracking.plots import map_vs_gpu_time
seed_everything(42)

In [ ]:
# 1. Training commands:
# !python scripts/train_ultra.py --config configs/rtdetr/e0_base.yaml --eval-val
# !python scripts/train_ultra.py --config configs/rtdetr/e1_queries.yaml --eval-val
# !python scripts/train_ultra.py --config configs/rtdetr/e2_matching_dn.yaml --eval-val

In [ ]:
# 2. Display F4 vs F3 comparison and plot map_vs_gpu_time_all.png
if P.results_csv.exists():
    df = pd.read_csv(P.results_csv)
    display(df[df['family'].isin(['rtdetr', 'yolo'])][['run_id', 'family', 'exp', 'AP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR500', 'fps', 'gpu_h', 'epochs_done']])
runs_dict = {}
for rid in ['yolo-e0', 'rtdetr-e0', 'rtdetr-e1', 'rtdetr-e2']:
    mp = P.exp_runs_dir / rid / 'metrics.csv'
    if mp.exists():
        runs_dict[rid] = pd.read_csv(mp)
if len(runs_dict) >= 2:
    map_vs_gpu_time(runs_dict, P.images / 'map_vs_gpu_time_all.png')